### 1. Descargar el dataset de Kaggle
Usaremos `kagglehub` para descargar el dataset directamente de manera pública y sin necesidad de credenciales o claves API.

In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("miadul/smart-crop-recommendation-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'smart-crop-recommendation-dataset' dataset.
Path to dataset files: /kaggle/input/smart-crop-recommendation-dataset


### 2. Carga y preprocesamiento de los datos
Cargaremos el archivo CSV, analizaremos sus columnas y las prepararemos para el entrenamiento del modelo.

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import os

# Localizar el archivo CSV en la ruta descargada por kagglehub
# 'path' viene de la celda de kagglehub ejecutada previamente
try:
    dataset_dir = path + "/data"
except NameError:
    dataset_dir = '.'

csv_files = [os.path.join(dataset_dir, f) for f in os.listdir(dataset_dir) if f.endswith('.csv')]
if not csv_files:
    raise FileNotFoundError("No se encontró el archivo CSV del dataset en la ruta de kagglehub.")

df = pd.read_csv(csv_files[0])
print(f"Dataset cargado con éxito desde: {csv_files[0]}")
print("\nColumnas disponibles en el dataset:")
print(df.columns.tolist())
display(df.head())


Dataset cargado con éxito desde: /kaggle/input/smart-crop-recommendation-dataset/data/crop-yield.csv

Columnas disponibles en el dataset:
['N', 'P', 'K', 'Soil_pH', 'Soil_Moisture', 'Soil_Type', 'Organic_Carbon', 'Temperature', 'Humidity', 'Rainfall', 'Sunlight_Hours', 'Wind_Speed', 'Region', 'Altitude', 'Season', 'Crop_Type', 'Irrigation_Type', 'Fertilizer_Used', 'Pesticide_Used', 'Crop_Yield_ton_per_hectare']


,N,P,K,Soil_pH,Soil_Moisture,Soil_Type,Organic_Carbon,Temperature,Humidity,Rainfall,Sunlight_Hours,Wind_Speed,Region,Altitude,Season,Crop_Type,Irrigation_Type,Fertilizer_Used,Pesticide_Used,Crop_Yield_ton_per_hectare
0,132,62,22,6.35,59.78,Clay,0.43,22.97,53.89,1305.68,7.73,15.96,Central,36,Rabi,Maize,Canal,223.48,23.36,11.42
1,122,71,66,5.98,25.54,Sandy,0.65,17.00,76.90,1942.05,9.25,12.60,North,1561,Rabi,Potato,Canal,161.54,4.42,23.19
2,44,35,104,8.07,25.87,Sandy,0.79,25.52,44.78,2216.20,8.50,15.63,North,1870,Rabi,Rice,Rainfed,184.62,6.29,7.94
3,136,96,113,4.83,42.97,Silt,0.45,18.59,31.89,607.18,8.75,5.49,East,765,Kharif,Sugarcane,Rainfed,274.02,2.72,72.53
4,101,34,42,5.84,48.01,Silt,0.69,22.74,46.27,483.47,8.00,7.44,Central,1143,Zaid,Wheat,Rainfed,72.69,15.37,6.72


### 3. Entrenamiento del Modelo de Recomendación de Cultivos
Entrenaremos un modelo que aprenda a predecir la productividad ('Yield') para diferentes tipos de cultivos dadas las condiciones del suelo y clima.

In [14]:
# Mapeo estimado de columnas basado en el dataset típico de agricultura inteligente.
# Ajustaremos los nombres de las columnas para que coincidan de forma segura.
column_mapping = {
    'Temperature': [c for c in df.columns if 'temperature' in c.lower()][0],
    'Humidity': [c for c in df.columns if 'humidity' in c.lower()][0],
    'Soil_Moisture': [c for c in df.columns if 'soil_moisture' in c.lower()][0],
    'Sunlight_Hours': [c for c in df.columns if 'sunlight_hours' in c.lower()][0],
    'Soil_pH': [c for c in df.columns if 'soil_ph' in c.lower()][0],
    'Crop': [c for c in df.columns if 'crop_type' in c.lower()][0],
    'Yield': [c for c in df.columns if 'crop_yield_ton_per_hectare' in c.lower()][0]
}

print("Mapeo de columnas detectado:")
for k, v in column_mapping.items():
    print(f"- {k} -> {v}")

# Preparando variables X e y
features = [column_mapping['Temperature'], column_mapping['Humidity'], column_mapping['Soil_Moisture'], column_mapping['Sunlight_Hours'], column_mapping['Soil_pH']]
target_crop = column_mapping['Crop']
target_yield = column_mapping['Yield']

# Codificación One-Hot de los cultivos para predecir el rendimiento específico por cultivo
X = pd.get_dummies(df[features + [target_crop]], columns=[target_crop])
y = df[target_yield]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print(f"Modelo entrenado con un score R² de prueba de: {model.score(X_test, y_test):.4f}")

Mapeo de columnas detectado:
- Temperature -> Temperature
- Humidity -> Humidity
- Soil_Moisture -> Soil_Moisture
- Sunlight_Hours -> Sunlight_Hours
- Soil_pH -> Soil_pH
- Crop -> Crop_Type
- Yield -> Crop_Yield_ton_per_hectare
Modelo entrenado con un score R² de prueba de: 0.9953


### 4. Función de Predicción del Mejor Cultivo
Esta función simula el rendimiento de cada posible cultivo bajo tus condiciones actuales de sensor y te dice cuál te dará mayor productividad.

In [15]:
def recomendar_cultivo(temperatura, humedad_amb, humedad_tierra, horas_luz, ph):
    cultivos_disponibles = df[target_crop].unique()
    resultados = []

    for crop in cultivos_disponibles:
        # Crear un registro con las condiciones del sensor más el cultivo evaluado
        data_dict = {
            column_mapping['Temperature']: temperatura,
            column_mapping['Humidity']: humedad_amb,
            column_mapping['Soil_Moisture']: humedad_tierra,
            column_mapping['Sunlight_Hours']: horas_luz,
            column_mapping['Soil_pH']: ph,
        }

        # Inicializar dummies de cultivos en 0
        for c in cultivos_disponibles:
            col_dummy = f"{target_crop}_{c}"
            data_dict[col_dummy] = 1 if c == crop else 0

        input_df = pd.DataFrame([data_dict])[X.columns]
        pred_yield = model.predict(input_df)[0]
        resultados.append((crop, pred_yield))

    resultados.sort(key=lambda x: x[1], reverse=True)

    print("--- RENDIMIENTO ESTIMADO POR CULTIVO ---")
    for i, (c, y_val) in enumerate(resultados):
        print(f"{i+1}. {c}: {y_val:.2f} (Productividad estimada)")

    print(f"\n El cultivo recomendado para obtener la mayor productividad es: **{resultados[0][0].upper()}**")

# Prueba la función con tus valores (ejemplo):
recomendar_cultivo(temperatura=25.0, humedad_amb=30.0, humedad_tierra=50.0, horas_luz=8.0, ph=6.0)

--- RENDIMIENTO ESTIMADO POR CULTIVO ---
1. Sugarcane: 75.75 (Productividad estimada)
2. Potato: 25.77 (Productividad estimada)
3. Maize: 10.41 (Productividad estimada)
4. Rice: 9.86 (Productividad estimada)
5. Wheat: 8.84 (Productividad estimada)
6. Cotton: 8.17 (Productividad estimada)

 El cultivo recomendado para obtener la mayor productividad es: **SUGARCANE**


In [18]:
# 1. Revisar si el modelo memorizó el entrenamiento (Overfitting)
print(f"R² en Entrenamiento: {model.score(X_train, y_train):.4f}")
print(f"R² en Prueba:        {model.score(X_test, y_test):.4f}")

# 2. Ver la correlación real de TODAS las variables numéricas con el rendimiento
print("\nCorrelación con Crop_Yield_ton_per_hectare:")
print(df.corr(numeric_only=True)['Crop_Yield_ton_per_hectare'].sort_values(ascending=False))

R² en Entrenamiento: 0.9993
R² en Prueba:        0.9953

Correlación con Crop_Yield_ton_per_hectare:
Crop_Yield_ton_per_hectare    1.000000
Fertilizer_Used               0.051371
Rainfall                      0.031213
Wind_Speed                    0.015185
K                             0.009897
Humidity                      0.006916
N                             0.002334
Soil_Moisture                 0.000152
Organic_Carbon               -0.000781
Sunlight_Hours               -0.001162
Altitude                     -0.001562
P                            -0.002483
Temperature                  -0.006589
Soil_pH                      -0.010509
Pesticide_Used               -0.012060
Name: Crop_Yield_ton_per_hectare, dtype: float64


In [20]:
print(df.groupby('Crop_Type')['Crop_Yield_ton_per_hectare'].agg(['mean', 'std']))

                mean       std
Crop_Type                     
Cotton      7.427941  1.727750
Maize      10.076037  1.701917
Potato     24.891012  1.681934
Rice        9.402027  1.657081
Sugarcane  74.830257  1.637562
Wheat       8.724680  1.649251


### 5. Exportar el Modelo y Metadatos
Guardaremos el modelo entrenado junto con las columnas esperadas utilizando `joblib` para que puedas descargarlo e importarlo fácilmente en cualquier otro script de Python.

In [21]:
import joblib

# Definir los nombres de los archivos para exportar
model_filename = 'modelo_rendimiento_cultivos.joblib'
metadata_filename = 'metadatos_modelo.joblib'

# 1. Guardar el modelo entrenado
joblib.dump(model, model_filename)

# 2. Guardar metadatos esenciales para la inferencia posterior
metadata = {
    'features': features,
    'target_crop': target_crop,
    'target_yield': target_yield,
    'column_mapping': column_mapping,
    'model_columns': list(X.columns),
    'cultivos_disponibles': list(df[target_crop].unique())
}
joblib.dump(metadata, metadata_filename)

print(f"¡Éxito! Modelo guardado como: {model_filename}")
print(f"Metadatos guardados como: {metadata_filename}")

¡Éxito! Modelo guardado como: modelo_rendimiento_cultivos.joblib
Metadatos guardados como: metadatos_modelo.joblib


#### Ejemplo de cómo cargar el modelo y hacer inferencia en otro lugar
Aquí tienes un ejemplo rápido de cómo cargarías estos archivos exportados para realizar una recomendación de cultivo desde cero.

In [22]:
# Código de ejemplo para cargar y usar el modelo exportado:
import joblib
import pandas as pd

# 1. Cargar el modelo y los metadatos
modelo_cargado = joblib.load('modelo_rendimiento_cultivos.joblib')
meta = joblib.load('metadatos_modelo.joblib')

# 2. Definir función de recomendación usando únicamente los archivos exportados
def recomendar_con_modelo_exportado(temperatura, humedad_amb, humedad_tierra, horas_luz, ph):
    resultados = []
    for crop in meta['cultivos_disponibles']:
        data_dict = {
            meta['column_mapping']['Temperature']: temperatura,
            meta['column_mapping']['Humidity']: humedad_amb,
            meta['column_mapping']['Soil_Moisture']: humedad_tierra,
            meta['column_mapping']['Sunlight_Hours']: horas_luz,
            meta['column_mapping']['Soil_pH']: ph,
        }

        for c in meta['cultivos_disponibles']:
            col_dummy = f"{meta['target_crop']}_{c}"
            data_dict[col_dummy] = 1 if c == crop else 0

        input_df = pd.DataFrame([data_dict])[meta['model_columns']]
        pred_yield = modelo_cargado.predict(input_df)[0]
        resultados.append((crop, pred_yield))

    resultados.sort(key=lambda x: x[1], reverse=True)
    return resultados

# Ejemplo de uso:
print(recomendar_con_modelo_exportado(25.0, 30.0, 50.0, 8.0, 6.0)[0])

('Sugarcane', np.float64(75.75119999999998))


### 6. Función de Inferencia Avanzada (Top 10 e Inferencia Individual)

Esta función mejorada te permite obtener el ranking completo de rendimiento de cultivos o consultar la predicción exacta de un solo cultivo específico.

In [23]:
def recomendacion_avanzada(temperatura, humedad_amb, humedad_tierra, horas_luz, ph, cultivo_especifico=None):
    # Obtener todas las predicciones para cada cultivo disponible
    todas_predicciones = recomendar_con_modelo_exportado(temperatura, humedad_amb, humedad_tierra, horas_luz, ph)

    # Caso 1: El usuario quiere saber la predicción de un cultivo específico
    if cultivo_especifico:
        # Normalizar el formato del texto ingresado
        cultivo_buscado = cultivo_especifico.strip().capitalize()
        prediccion = next((prod for crop, prod in todas_predicciones if crop.capitalize() == cultivo_buscado), None)

        if prediccion is not None:
            print(f"--- PREDICCIÓN INDIVIDUAL ---")
            print(f"La productividad estimada para el cultivo '{cultivo_buscado}' es de: {prediccion:.2f} toneladas por hectárea.\n")
            return (cultivo_buscado, prediccion)
        else:
            print(f"El cultivo '{cultivo_especifico}' no se encuentra en el dataset.")
            print(f"Cultivos disponibles: {meta['cultivos_disponibles']}\n")
            return None

    # Caso 2: El usuario quiere el ranking de los mejores cultivos (Top 10)
    else:
        print(f"--- TOP MEJORES CULTIVOS ---")
        for rank, (crop, yield_val) in enumerate(todas_predicciones[:10], start=1):
            print(f"{rank:2d}. {crop:<12} -> Rendimiento estimado: {yield_val:.2f} t/ha")
        print()
        return todas_predicciones[:10]

# --- DEMOSTRACIONES DE USO ---

# Demostración 1: Obtener el ranking (Top 10)
ranking = recomendacion_avanzada(temperatura=25.0, humedad_amb=30.0, humedad_tierra=50.0, horas_luz=8.0, ph=6.0)

# Demostración 2: Consultar la predicción de un solo cultivo en específico (ej. 'Wheat')
pred_individual = recomendacion_avanzada(temperatura=25.0, humedad_amb=30.0, humedad_tierra=50.0, horas_luz=8.0, ph=6.0, cultivo_especifico='Wheat')

--- TOP MEJORES CULTIVOS ---
 1. Sugarcane    -> Rendimiento estimado: 75.75 t/ha
 2. Potato       -> Rendimiento estimado: 25.77 t/ha
 3. Maize        -> Rendimiento estimado: 10.41 t/ha
 4. Rice         -> Rendimiento estimado: 9.86 t/ha
 5. Wheat        -> Rendimiento estimado: 8.84 t/ha
 6. Cotton       -> Rendimiento estimado: 8.17 t/ha

--- PREDICCIÓN INDIVIDUAL ---
La productividad estimada para el cultivo 'Wheat' es de: 8.84 toneladas por hectárea.

